In [1]:
from datetime import date, timedelta
from dateutil.rrule import rrule, DAILY

import itertools

import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
%matplotlib inline

import numpy as np
import pandas as pd

import scipy.stats as ss
from sklearn import linear_model
import statsmodels.api as sm

import warnings

test_count = 7

In [2]:
df_vitals_combined = pd.read_csv('vitals_combined_interpolated.csv')

In [3]:
# Drops batches containing less than 30 observations.
df_recordings_per_patient = df_vitals_combined.groupby(['patient_uuid', 'batch']).count().reset_index()
low_batches = df_recordings_per_patient[(df_recordings_per_patient.HR <= 30)][['patient_uuid','batch']]

for i in range(len(low_batches)):
    low_patient = low_batches.patient_uuid.iloc[i]
    low_batch = low_batches.batch.iloc[i]
    df_vitals_combined.drop(df_vitals_combined[(df_vitals_combined.patient_uuid == low_patient) & (df_vitals_combined.batch == low_batch)].index, inplace=True)

In [4]:
print(df_vitals_combined.shape)
idx = df_vitals_combined.groupby(['patient_uuid'])['batch'].transform(max) == df_vitals_combined['batch']

df_vitals_combined = df_vitals_combined[idx]

print(df_vitals_combined.shape)

(121156, 5)
(111777, 5)


In [5]:
df_vitals_combined_SBP = df_vitals_combined[['patient_uuid','reading_date','SBP']]

df_vitals_combined_HR = df_vitals_combined[['patient_uuid','reading_date','HR']]

In [6]:
# Define the p, d and q parameters to take any value between 0 and 2
p = d = q = range(0, 2)

# Generate all different combinations of p, q and q triplets
pdq = list(itertools.product(p, d, q))

# Generate all different combinations of seasonal p, q and q triplets
seasonal_pdq = [(x[0], x[1], x[2], 12) for x in list(itertools.product(p, d, q))]

print('Examples of parameter combinations for Seasonal ARIMA...')
print('SARIMAX: {} x {}'.format(pdq[1], seasonal_pdq[1]))
print('SARIMAX: {} x {}'.format(pdq[1], seasonal_pdq[2]))
print('SARIMAX: {} x {}'.format(pdq[2], seasonal_pdq[3]))
print('SARIMAX: {} x {}'.format(pdq[2], seasonal_pdq[4]))

Examples of parameter combinations for Seasonal ARIMA...
SARIMAX: (0, 0, 1) x (0, 0, 1, 12)
SARIMAX: (0, 0, 1) x (0, 1, 0, 12)
SARIMAX: (0, 1, 0) x (0, 1, 1, 12)
SARIMAX: (0, 1, 0) x (1, 0, 0, 12)


In [7]:
warnings.filterwarnings("ignore") # specify to ignore warning messages

ARIMA_gridsearch_stats = pd.DataFrame(columns = ['patient_uuid','p','d','q','P','D','Q','S','AIC'])
currloop = 0
loopcount = df_vitals_combined_SBP['patient_uuid'].unique().shape[0]
for patient_uuid in df_vitals_combined_SBP['patient_uuid'].unique():
    print(str(currloop) + ' out of ' + str(loopcount) + ', patient_uuid = ' + str(patient_uuid) )
    currloop = currloop + 1
    df_vitals_subset_batch = df_vitals_combined_SBP[df_vitals_combined_SBP['patient_uuid'] == patient_uuid]
    df_vitals_subset_batch = df_vitals_subset_batch[['reading_date','SBP']]
    df_vitals_subset_batch.set_index('reading_date',inplace=True)
    train_count = int(np.round(df_vitals_subset_batch.shape[0])) - 7
    df_vitals_train = df_vitals_subset_batch[0:train_count]
    df_vitals_test = df_vitals_subset_batch[train_count:]
    print(df_vitals_train.shape)
    for param in pdq:
        for param_seasonal in seasonal_pdq:
            try:
                mod = sm.tsa.statespace.SARIMAX(df_vitals_train,
                                            order=param,
                                            seasonal_order=param_seasonal,
                                            enforce_stationarity=False,
                                            enforce_invertibility=False)

                results = mod.fit()

                ARIMA_gridsearch_stats.loc[len(ARIMA_gridsearch_stats)] = [int(patient_uuid),param[0],param[1],param[2],param_seasonal[0],param_seasonal[1],param_seasonal[2],param_seasonal[3], results.aic]
                #print('ARIMA{}x{}12 - AIC:{}'.format(param, param_seasonal, results.aic))
            except:
                continue
    #if currloop > 5:
        #break

print('ARIMA grid search completed')

0 out of 883, patient_uuid = 1127912
(48, 1)
1 out of 883, patient_uuid = 1164638
(125, 1)
2 out of 883, patient_uuid = 1166047
(54, 1)
3 out of 883, patient_uuid = 1193821
(76, 1)
4 out of 883, patient_uuid = 1231804
(135, 1)
5 out of 883, patient_uuid = 1519674
(98, 1)
6 out of 883, patient_uuid = 1523219
(42, 1)
7 out of 883, patient_uuid = 2075791
(25, 1)
8 out of 883, patient_uuid = 2096100
(31, 1)
9 out of 883, patient_uuid = 2104097
(99, 1)
10 out of 883, patient_uuid = 2132729
(92, 1)
11 out of 883, patient_uuid = 2168037
(119, 1)
12 out of 883, patient_uuid = 2172350
(381, 1)
13 out of 883, patient_uuid = 2469101
(46, 1)
14 out of 883, patient_uuid = 2714491
(169, 1)
15 out of 883, patient_uuid = 2781252
(52, 1)
16 out of 883, patient_uuid = 2793778
(51, 1)
17 out of 883, patient_uuid = 2802493
(374, 1)
18 out of 883, patient_uuid = 2823949
(28, 1)
19 out of 883, patient_uuid = 3005945
(59, 1)
20 out of 883, patient_uuid = 3224753
(63, 1)
21 out of 883, patient_uuid = 3371769


In [8]:
ARIMA_gridsearch_stats.head(5)

,patient_uuid,p,d,q,P,D,Q,S,AIC
0,1127912.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0,591.992912
1,1127912.0,0.0,0.0,0.0,0.0,0.0,1.0,12.0,1342.304881
2,1127912.0,0.0,0.0,0.0,0.0,1.0,0.0,12.0,255.762693
3,1127912.0,0.0,0.0,0.0,1.0,0.0,0.0,12.0,263.388566
4,1127912.0,0.0,0.0,0.0,1.0,0.0,1.0,12.0,245.939801


In [9]:
idx = ARIMA_gridsearch_stats.groupby(['patient_uuid'])['AIC'].transform(min) == ARIMA_gridsearch_stats['AIC']

ARIMA_gridsearch_stats_min = ARIMA_gridsearch_stats[idx]

ARIMA_gridsearch_stats_min

,patient_uuid,p,d,q,P,D,Q,S,AIC
47,1127912.0,1.0,1.0,1.0,1.0,1.0,0.0,12.0,156.692762
79,1164638.0,0.0,1.0,1.0,1.0,1.0,1.0,12.0,645.997576
174,1166047.0,1.0,1.0,1.0,1.0,1.0,0.0,12.0,187.670001
207,1193821.0,0.0,1.0,1.0,1.0,1.0,1.0,12.0,351.049833
271,1231804.0,0.0,1.0,1.0,1.0,1.0,1.0,12.0,912.344502
...,...,...,...,...,...,...,...,...,...
50600,98772423.0,1.0,1.0,1.0,0.0,1.0,1.0,12.0,641.055071
50632,98999464.0,0.0,1.0,1.0,0.0,1.0,1.0,12.0,328.421203
50696,99535439.0,0.0,1.0,1.0,0.0,1.0,1.0,12.0,293.714103
50795,99683668.0,1.0,1.0,1.0,1.0,1.0,0.0,12.0,172.949938


In [10]:
currloop = 0
#loopcount = ARIMA_gridsearch_stats_min['patient_uuid'].unique().shape[0]
loopcount = df_vitals_combined_SBP['patient_uuid'].unique().shape[0]
df_vitals_combined_SBP_predicted = pd.DataFrame(columns = ['patient_uuid','reading_date','SBP'])
df_vitals_combined_SBP_forecast = pd.DataFrame(columns = ['patient_uuid','reading_date','SBP'])
#for patient_uuid in ARIMA_gridsearch_stats_min['patient_uuid'].unique():
for patient_uuid in df_vitals_combined_SBP['patient_uuid'].unique():
    print(str(currloop) + ' out of ' + str(loopcount) + ', patient_uuid = ' + str(patient_uuid) )
    currloop = currloop + 1
    ARIMA_gridsearch_stats_min_patient = ARIMA_gridsearch_stats_min[ARIMA_gridsearch_stats_min['patient_uuid'] == patient_uuid]
    p = int(ARIMA_gridsearch_stats_min_patient.iloc[0]['p'])
    d = int(ARIMA_gridsearch_stats_min_patient.iloc[0]['d'])
    q = int(ARIMA_gridsearch_stats_min_patient.iloc[0]['q'])
    P = int(ARIMA_gridsearch_stats_min_patient.iloc[0]['P'])
    D = int(ARIMA_gridsearch_stats_min_patient.iloc[0]['D'])
    Q = int(ARIMA_gridsearch_stats_min_patient.iloc[0]['Q'])
    S = int(ARIMA_gridsearch_stats_min_patient.iloc[0]['S'])

    df_vitals_subset_batch = df_vitals_combined_SBP[df_vitals_combined_SBP['patient_uuid'] == patient_uuid]
    df_vitals_subset_batch = df_vitals_subset_batch[['reading_date','SBP']]
    df_vitals_subset_batch.set_index('reading_date',inplace=True)
    train_count = int(np.round(df_vitals_subset_batch.shape[0])) - 7
    test_count = 7
    df_vitals_train = df_vitals_subset_batch[0:train_count]
    df_vitals_test = df_vitals_subset_batch[train_count:]
    try:
        pred_start_date = (pd.to_datetime(list(df_vitals_test.index.values)[-test_count])).date()
        pred_end_date = (pd.to_datetime(pred_start_date) + timedelta(days=test_count-1)).date()
                           
        mod = sm.tsa.statespace.SARIMAX(df_vitals_train,
                                    order=(p, d, q),
                                    seasonal_order=(P, D, Q, S),
                                    enforce_stationarity=False,
                                    enforce_invertibility=False)

        results = mod.fit()

        pred = results.get_prediction(start=pred_start_date,end=pred_end_date,dynamic=False)
        pred_df = pd.DataFrame(pred.predicted_mean)
        pred_df.columns=['SBP']
        pred_df['reading_date'] = pred_df.index
        pred_df['patient_uuid'] = np.repeat(patient_uuid,pred_df.shape[0])
        df_vitals_combined_SBP_predicted = df_vitals_combined_SBP_predicted.append(pred_df)
       
        forecast_start_date = pred_end_date + timedelta(days=1)
        forecast_end_date = (pd.to_datetime(forecast_start_date) + timedelta(days=test_count-1)).date()
        
        forecast = results.get_prediction(start=forecast_start_date,end=forecast_end_date ,dynamic=False)
        forecast_df = pd.DataFrame(forecast.predicted_mean)
        forecast_df.columns=['SBP']
        forecast_df['reading_date'] = forecast_df.index
        forecast_df['patient_uuid'] = np.repeat(patient_uuid,forecast_df.shape[0])
        df_vitals_combined_SBP_forecast = df_vitals_combined_SBP_forecast.append(forecast_df)
    
    except:
        print('exception')
        pass

print('Forecast Procedure completed')


0 out of 883, patient_uuid = 1127912
1 out of 883, patient_uuid = 1164638
2 out of 883, patient_uuid = 1166047
3 out of 883, patient_uuid = 1193821
4 out of 883, patient_uuid = 1231804
5 out of 883, patient_uuid = 1519674
6 out of 883, patient_uuid = 1523219
7 out of 883, patient_uuid = 2075791
8 out of 883, patient_uuid = 2096100
9 out of 883, patient_uuid = 2104097
10 out of 883, patient_uuid = 2132729
11 out of 883, patient_uuid = 2168037
12 out of 883, patient_uuid = 2172350
13 out of 883, patient_uuid = 2469101
14 out of 883, patient_uuid = 2714491
15 out of 883, patient_uuid = 2781252
16 out of 883, patient_uuid = 2793778
17 out of 883, patient_uuid = 2802493
18 out of 883, patient_uuid = 2823949
19 out of 883, patient_uuid = 3005945
20 out of 883, patient_uuid = 3224753
21 out of 883, patient_uuid = 3371769
22 out of 883, patient_uuid = 3415879
23 out of 883, patient_uuid = 3429852
24 out of 883, patient_uuid = 3469162
exception
25 out of 883, patient_uuid = 3501786
26 out of 88

In [11]:
df_vitals_combined_SBP_predicted.to_csv("df_vitals_combined_SBP_predicted.csv", index = False)

In [12]:
df_vitals_combined_SBP_forecast.to_csv("df_vitals_combined_SBP_forecast.csv", index = False)